In [9]:
import re
import pandas as pd
from difflib import get_close_matches

# -------------------------------
# Levenshtein Distance
# -------------------------------
try:
    from Levenshtein import distance as levenshtein_distance
except:
    def levenshtein_distance(a, b):
        if len(a) < len(b):
            return levenshtein_distance(b, a)
        if len(b) == 0:
            return len(a)
        previous_row = range(len(b) + 1)
        for i, c1 in enumerate(a):
            current_row = [i + 1]
            for j, c2 in enumerate(b):
                insertions = previous_row[j + 1] + 1
                deletions = current_row[j] + 1
                substitutions = previous_row[j] + (c1 != c2)
                current_row.append(min(insertions, deletions, substitutions))
            previous_row = current_row
        return previous_row[-1]


# -------------------------------
# Load Data
# -------------------------------
with open("word.txt", "r", encoding="utf-8") as f:
    words = list(set([w.strip() for w in f.readlines()]))

with open("hindi_dict.txt", "r", encoding="utf-8") as f:
    hindi_dict = set([w.strip() for w in f.readlines()])


# -------------------------------
# Normalization
# -------------------------------
def normalize(word):
    return re.sub(r'[^\u0900-\u097F]', '', word).strip()


COMMON_MISTAKES = {
    "नही": "नहीं",
    "मै": "मैं",
    "क्यूंकि": "क्योंकि",
    "बोहोत": "बहुत",
    "वहा": "वहाँ",
    "हैना": "है ना",
    "अलगअलग": "अलग-अलग",
    "बच्पन": "बचपन"
}


def phonetic_simplify(word):
    replacements = {"ौ": "ो", "ै": "े", "ं": "", "ँ": ""}
    for k, v in replacements.items():
        word = word.replace(k, v)
    return word


# -------------------------------
# 🔥 Fine-tuned Whisper Model (ONLY for ASR)
# -------------------------------
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq
import torch

processor = AutoProcessor.from_pretrained("model_final")
asr_model = AutoModelForSpeechSeq2Seq.from_pretrained("model_final")

asr_model.eval()


def transcribe(audio_array, sampling_rate=16000):
    inputs = processor(audio_array, sampling_rate=sampling_rate, return_tensors="pt")

    with torch.no_grad():
        generated_ids = asr_model.generate(inputs["input_features"])

    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]


# -------------------------------
# Classification (RULE-BASED ONLY)
# -------------------------------
def classify_word(word):
    clean = normalize(word)

    if not clean:
        return "Incorrect", "Low", "Empty"

    if clean in hindi_dict:
        return "Correct", "High", "Dictionary match"

    if clean in COMMON_MISTAKES:
        return "Incorrect", "High", f"Fix → {COMMON_MISTAKES[clean]}"

    if "_" in word or any(c.isdigit() for c in word):
        return "Incorrect", "High", "Invalid characters"

    matches = get_close_matches(clean, hindi_dict, n=1, cutoff=0.75)
    if matches:
        closest = matches[0]
        dist = levenshtein_distance(clean, closest)

        if dist <= 1:
            return "Incorrect", "High", f"Typo → {closest}"
        elif dist <= 2:
            return "Incorrect", "Medium", f"Close → {closest}"

    phon_clean = phonetic_simplify(clean)
    for w in list(hindi_dict)[:3000]:
        if phonetic_simplify(w) == phon_clean:
            return "Incorrect", "Medium", f"Phonetic → {w}"

    return "Incorrect", "Low", "Unknown"


# -------------------------------
# Run Pipeline (TEXT INPUT)
# -------------------------------
results = []

for i, word in enumerate(words):
    label, confidence, reason = classify_word(word)
    results.append([word, label, confidence, reason])

    if i % 10000 == 0:
        print(f"Processed {i} words...")


df = pd.DataFrame(results, columns=["Word", "Label", "Confidence", "Reason"])
df.to_csv("final_output.csv", index=False, encoding="utf-8")


# -------------------------------
# Summary
# -------------------------------
print("\n📊 Summary:")
print(df["Label"].value_counts())

print("\n📊 Confidence Distribution:")
print(df["Confidence"].value_counts())

print("\n✅ Done! File saved as final_output.csv")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Processed 0 words...
Processed 10000 words...
Processed 20000 words...
Processed 30000 words...
Processed 40000 words...
Processed 50000 words...
Processed 60000 words...
Processed 70000 words...
Processed 80000 words...
Processed 90000 words...
Processed 100000 words...
Processed 110000 words...
Processed 120000 words...
Processed 130000 words...
Processed 140000 words...
Processed 150000 words...
Processed 160000 words...
Processed 170000 words...

📊 Summary:
Label
Incorrect    177198
Correct         189
Name: count, dtype: int64

📊 Confidence Distribution:
Confidence
Low       173699
High        2623
Medium      1065
Name: count, dtype: int64

✅ Done! File saved as final_output.csv


In [13]:
!pip install reportlab

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   --------------------- ------------------ 1.0/2.0 MB 7.2 MB/s eta 0:00:01
   -------------------------------- ------- 1.6/2.0 MB 5.4 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 4.4 MB/s  0:00:00



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
